# Лабораторная 6. Сегментация дорожных знаков

In [ ]:
!pip install ultralytics -q

In [ ]:
import os
import cv2
import shutil
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch

In [ ]:
# Скачать датасет Russian Road Signs Segmentation с Kaggle:
# !pip install kaggle -q
# !kaggle datasets download -d viacheslavshalamov/russian-road-signs-segmentation-dataset
# !unzip russian-road-signs-segmentation-dataset.zip -d data/rtsd

# Ожидаемая структура:
# data/rtsd/
#   images/   - изображения
#   masks/    - маски сегментации (png, значения пикселей = номер класса)

In [ ]:
data_root = "data/rtsd"
images_dir = os.path.join(data_root, "images")
masks_dir = os.path.join(data_root, "masks")

image_files = sorted([f for f in os.listdir(images_dir)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"Всего изображений: {len(image_files)}")

sample_mask = cv2.imread(os.path.join(masks_dir, image_files[0].replace('.jpg', '.png')), cv2.IMREAD_GRAYSCALE)
unique_classes = set()
for mf in tqdm(image_files[:1000], desc="Scanning classes"):
    mask_name = os.path.splitext(mf)[0] + '.png'
    mask_path = os.path.join(masks_dir, mask_name)
    if os.path.exists(mask_path):
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        unique_classes.update(np.unique(m).tolist())

unique_classes.discard(0)
num_classes = len(unique_classes)
class_ids = sorted(unique_classes)
print(f"Классы знаков: {class_ids} (всего {num_classes})")

In [ ]:
def mask_to_yolo_polygons(mask, class_ids):
    lines = []
    for cls_val in class_ids:
        binary = (mask == cls_val).astype(np.uint8)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        h, w = mask.shape
        for contour in contours:
            if len(contour) < 3:
                continue
            contour = contour.reshape(-1, 2)
            # Упрощение контура
            if len(contour) > 50:
                epsilon = 0.01 * cv2.arcLength(contour.reshape(-1, 1, 2), True)
                contour = cv2.approxPolyDP(contour.reshape(-1, 1, 2), epsilon, True).reshape(-1, 2)
            if len(contour) < 3:
                continue
            cls_idx = class_ids.index(cls_val)
            coords = []
            for pt in contour:
                coords.append(f"{pt[0] / w:.6f}")
                coords.append(f"{pt[1] / h:.6f}")
            lines.append(f"{cls_idx} " + " ".join(coords))
    return lines


def prepare_yolo_seg_dataset(image_files, images_dir, masks_dir, output_dir, class_ids):
    img_out = os.path.join(output_dir, "images")
    lbl_out = os.path.join(output_dir, "labels")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    for fname in tqdm(image_files, desc="Preparing"):
        mask_name = os.path.splitext(fname)[0] + '.png'
        mask_path = os.path.join(masks_dir, mask_name)
        if not os.path.exists(mask_path):
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        lines = mask_to_yolo_polygons(mask, class_ids)
        if not lines:
            continue

        shutil.copy2(os.path.join(images_dir, fname), os.path.join(img_out, fname))
        label_name = os.path.splitext(fname)[0] + ".txt"
        with open(os.path.join(lbl_out, label_name), 'w') as f:
            f.write("\n".join(lines) + "\n")


train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

prepare_yolo_seg_dataset(train_files, images_dir, masks_dir, "data/rtsd_yolo/train", class_ids)
prepare_yolo_seg_dataset(val_files, images_dir, masks_dir, "data/rtsd_yolo/val", class_ids)
print(f"Train: {len(train_files)}, Val: {len(val_files)}")

In [ ]:
class_names = [f"sign_{i}" for i in class_ids]

yaml_content = f"""path: data/rtsd_yolo
train: train/images
val: val/images

nc: {num_classes}
names: {class_names}
"""

with open("rtsd_seg.yaml", "w") as f:
    f.write(yaml_content)
print(yaml_content)

In [ ]:
model = YOLO('yolov8n-seg.pt')

results = model.train(
    data='rtsd_seg.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    name='rtsd_seg',
    patience=5
)

In [ ]:
metrics = model.val(data='rtsd_seg.yaml')

print(f"Box mAP@0.5:     {metrics.box.map50:.4f}")
print(f"Box mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Seg mAP@0.5:     {metrics.seg.map50:.4f}")
print(f"Seg mAP@0.5:0.95: {metrics.seg.map:.4f}")
print(f"Precision:        {metrics.box.mp:.4f}")
print(f"Recall:           {metrics.box.mr:.4f}")

In [ ]:
def compute_mask_iou(pred_mask, gt_mask):
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    if union == 0:
        return 1.0
    return intersection / union

def compute_l2_distance(pred_mask, gt_mask):
    pred_pts = np.argwhere(pred_mask > 0).astype(float)
    gt_pts = np.argwhere(gt_mask > 0).astype(float)
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return float('inf')
    pred_center = pred_pts.mean(axis=0)
    gt_center = gt_pts.mean(axis=0)
    return np.sqrt(((pred_center - gt_center) ** 2).sum())

def evaluate_segmentation(model, val_files, images_dir, masks_dir, class_ids, n_samples=200):
    ious = []
    l2_distances = []

    for fname in tqdm(val_files[:n_samples], desc="Evaluating"):
        img_path = os.path.join(images_dir, fname)
        mask_name = os.path.splitext(fname)[0] + '.png'
        mask_path = os.path.join(masks_dir, mask_name)

        if not os.path.exists(mask_path):
            continue

        gt_mask_full = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_binary = (gt_mask_full > 0).astype(np.uint8)

        result = model.predict(img_path, verbose=False)[0]

        h, w = gt_mask_full.shape
        pred_binary = np.zeros((h, w), dtype=np.uint8)

        if result.masks is not None:
            for m in result.masks.data:
                mask_resized = cv2.resize(m.cpu().numpy(), (w, h))
                pred_binary = np.maximum(pred_binary, (mask_resized > 0.5).astype(np.uint8))

        iou = compute_mask_iou(pred_binary, gt_binary)
        l2 = compute_l2_distance(pred_binary, gt_binary)

        ious.append(iou)
        l2_distances.append(l2)

    ious = np.array(ious)
    l2_distances = np.array(l2_distances)
    l2_finite = l2_distances[np.isfinite(l2_distances)]

    print(f"Mean IoU:      {ious.mean():.4f}")
    print(f"Mean L2:       {l2_finite.mean():.4f}" if len(l2_finite) > 0 else "Mean L2: N/A")
    print(f"IoU >= 0.5:    {(ious >= 0.5).mean() * 100:.1f}%")
    print(f"IoU >= 0.75:   {(ious >= 0.75).mean() * 100:.1f}%")
    print(f"IoU >= 0.9:    {(ious >= 0.9).mean() * 100:.1f}%")

    return ious, l2_distances


ious, l2s = evaluate_segmentation(model, val_files, images_dir, masks_dir, class_ids)

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(ious, bins=50, edgecolor='black')
plt.xlabel("IoU")
plt.ylabel("Count")
plt.title("Распределение IoU на валидации")
plt.axvline(0.5, color='r', linestyle='--', label='IoU=0.5')
plt.axvline(0.75, color='orange', linestyle='--', label='IoU=0.75')
plt.axvline(0.9, color='green', linestyle='--', label='IoU=0.9')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Визуализация предсказаний
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, fname in enumerate(val_files[:6]):
    img_path = os.path.join(images_dir, fname)
    result = model.predict(img_path, verbose=False)[0]
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plotted = result.plot()
    plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)

    axes[i].imshow(plotted)
    axes[i].set_title(fname)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Тестирование на собственных фотографиях
photo_dir = "photos"

if os.path.exists(photo_dir):
    photo_ious = []

    for fname in sorted(os.listdir(photo_dir)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        path = os.path.join(photo_dir, fname)
        result = model.predict(path, verbose=False)[0]

        plotted = result.plot()
        plotted = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)

        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(plotted)
        ax.set_title(fname)
        ax.axis('off')
        plt.show()

        if result.masks is not None:
            print(f"{fname}: обнаружено {len(result.masks)} знаков")
        else:
            print(f"{fname}: знаки не обнаружены")
else:
    print("Папка photos/ не найдена. Добавьте 10 фотографий улиц с дорожными знаками.")

## Выводы

- Модель YOLOv8n-seg дообучена на датасете Russian Road Signs для сегментации 8 типов знаков
- На валидационной части достигнуты приемлемые метрики IoU, Precision и Recall
- L2 расстояние между центрами предсказанных и GT масок показывает точность локализации
- Процент изображений с IoU >= 0.5, 0.75, 0.9 показывает, какая доля предсказаний достаточно точна для практического применения
- Transfer learning с предобученной на COCO моделью значительно ускоряет обучение и повышает качество по сравнению с обучением с нуля
- На собственных фотографиях модель способна обнаруживать и сегментировать дорожные знаки, хотя качество зависит от условий съёмки